# <font color='white'>1. Read Libraries<font>

In [22]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [23]:
# ============================================================================
# ADIM 1: ÇOK BASİT FEATURE ENGİNEERİNG (LAG YOK)
# ============================================================================


def prepare_basic_features(df_):
    """
    Sadece o haftanın bilgilerini kullan, geçmiş bilgi yok
    """
    df = df_.copy()
    
    # Datetime'a çevir
    df['week_start'] = pd.to_datetime(df['week_start'])
    df['customer_created_at'] = pd.to_datetime(df['customer_created_at'])
    
    # === ZAMAN FEATURELARI ===
    df['week'] = df['week_start'].dt.isocalendar().week.astype(int)
    df['month'] = df['week_start'].dt.month.astype(int)
    df['year'] = df['week_start'].dt.year.astype(int)
    df['day_of_year'] = df['week_start'].dt.dayofyear.astype(int)
    
    # === MÜŞTERİ FEATURELARI ===
    # Müşteri kaç gündür platformda?
    df['customer_age_days'] = (df['week_start'] - df['customer_created_at']).dt.days.clip(lower=0).fillna(0).astype(int)

    return df
    

In [24]:
# ============================================================================
# ADIM 2: TRAIN-VALIDATION SPLIT (ÇOK ÖNEMLİ!)
# ============================================================================

def split_train_val_test(df, val_weeks=4,test_weeks=4):
    """
    Train setinde:
    - week_start: O haftanın tarihi
    - Target_purchase_next_1w: BİR SONRAKİ hafta alınacak mı?

    - Train: Model eğitimi için
    - Validation: Hiperparametre tuning, model seçimi için
    - Test: Final performans ölçümü için (validation'a overfitting'i önler)
    
    Gerçek test seti (etiketli olmayan) → Canlı/Production
    
    Toplam 46 hafta:
    - Train: 38 hafta (ilk %82)
    - Validation: 4 hafta (sonraki %9)
    - Test: 4 hafta (son %9)
    """
    
    df = df.copy()
    
    # Sadece Target'ı olan satırları kullan (eğer NaN varsa)
    df_with_target = df[df['Target_purchase_next_1w'].notna()].copy()
    
    unique_weeks = np.sort(df_with_target['week_start'].unique())
    
    print("="*80)
    print("TRAIN-VALIDATION-TEST SPLIT (3 PARÇA)")
    print("="*80)
    print(f"\nToplam hafta sayısı: {len(unique_weeks)}")
    print(f"İlk hafta: {unique_weeks[0]}")
    print(f"Son hafta: {unique_weeks[-1]}")
    
    
    # Cutoff'ları hesapla
    test_cutoff = unique_weeks[-test_weeks]  # Son test_weeks hafta → test
    val_cutoff = unique_weeks[-(test_weeks + val_weeks)]  # Ondan önceki val_weeks → validation
    
    print(f"\nSplit stratejisi:")
    print(f"  Validation başlangıcı: {val_cutoff}")
    print(f"  Test başlangıcı: {test_cutoff}")
    
    # Maskeleri oluştur
    train_mask = df_with_target['week_start'] < val_cutoff
    val_mask = (df_with_target['week_start'] >= val_cutoff) & (df_with_target['week_start'] < test_cutoff)
    test_mask = df_with_target['week_start'] >= test_cutoff
    
    df_train = df_with_target[train_mask].copy()
    df_val = df_with_target[val_mask].copy()
    df_test = df_with_target[test_mask].copy()
    
    # İstatistikler
    print(f"\n{'='*80}")
    print("TRAIN SET:")
    print(f"  - Hafta sayısı: {len(df_train['week_start'].unique())}")
    print(f"  - Tarih aralığı: {df_train['week_start'].min()} → {df_train['week_start'].max()}")
    print(f"  - Satır sayısı: {len(df_train):,}")
    print(f"  - Oran: {len(df_train)/len(df_with_target)*100:.1f}%")
    
    print(f"\nVALIDATION SET:")
    print(f"  - Hafta sayısı: {len(df_val['week_start'].unique())}")
    print(f"  - Tarih aralığı: {df_val['week_start'].min()} → {df_val['week_start'].max()}")
    print(f"  - Satır sayısı: {len(df_val):,}")
    print(f"  - Oran: {len(df_val)/len(df_with_target)*100:.1f}%")
    
    print(f"\nTEST SET (Hold-out):")
    print(f"  - Hafta sayısı: {len(df_test['week_start'].unique())}")
    print(f"  - Tarih aralığı: {df_test['week_start'].min()} → {df_test['week_start'].max()}")
    print(f"  - Satır sayısı: {len(df_test):,}")
    print(f"  - Oran: {len(df_test)/len(df_with_target)*100:.1f}%")
    
    print(f"\n{'='*80}")
    print("NOT:")
    print("  - Train: Modeli eğitmek için")
    print("  - Validation: Hiperparametre optimize etmek için")
    print("  - Test: Final performansı ölçmek için (validation'a overfitting önleme)")
    print("  - Real Test (etiketli olmayan): Canlı tahmin için")
    print(f"{'='*80}")
    
    return df_train, df_val, df_test

In [25]:
# ============================================================================
# ADIM 3: KATEGORİK ENCODE
# ============================================================================

def encode_categorical_features(df_train, df_val, df_test_holdout, df_test_real=None):
    """
    Kategorik değişkenleri encode et
    
    Parametreler:
    - df_train: Eğitim seti
    - df_val: Validation seti
    - df_test_holdout: Test seti (etiketli, bizim ayırdığımız)
    - df_test_real: Gerçek test seti (etiketli olmayan, yarışma verisi)
    """
    
    # Düşük kardinalite → One-Hot
    low_cardinality_cols = ['customer_category', 'customer_status', 'grade_name', 'unit_name']
    low_cardinality_cols = [col for col in low_cardinality_cols if col in df_train.columns]
    
    # Yüksek kardinalite → Label Encoding
    high_cardinality_cols = ['customer_id', 'product_id']
    high_cardinality_cols = [col for col in high_cardinality_cols if col in df_train.columns]
    
    print("\n" + "="*80)
    print("ENCODING STRATEJİSİ")
    print("="*80)
    print(f"One-Hot Encoding: {low_cardinality_cols}")
    print(f"Label Encoding: {high_cardinality_cols}")
    
    # ========================================================================
    # ONE-HOT ENCODING
    # ========================================================================
    
    if low_cardinality_cols:
        print("\n--- ONE-HOT ENCODING ---")
        
        for col in low_cardinality_cols:
            unique_values = df_train[col].astype(str).unique()
            print(f"{col}: {len(unique_values)} değer")
            
            for val in unique_values:
                col_name = f"{col}_{val}"
                df_train[col_name] = (df_train[col].astype(str) == val).astype(int)
                df_val[col_name] = (df_val[col].astype(str) == val).astype(int)
                df_test_holdout[col_name] = (df_test_holdout[col].astype(str) == val).astype(int)
                
                if df_test_real is not None:
                    df_test_real[col_name] = (df_test_real[col].astype(str) == val).astype(int)
    
    # ========================================================================
    # LABEL ENCODING
    # ========================================================================
    
    encoders = {}
    
    if high_cardinality_cols:
        print("\n--- LABEL ENCODING ---")
        
        for col in high_cardinality_cols:
            encoder = LabelEncoder()
            encoder.fit(df_train[col].astype(str))
            
            print(f"{col}: {len(encoder.classes_)} değer")
            
            # Train
            df_train[f'{col}_encoded'] = encoder.transform(df_train[col].astype(str))
            
            # Validation
            df_val[f'{col}_encoded'] = df_val[col].astype(str).map(
                lambda x: encoder.transform([x])[0] if x in encoder.classes_ else -1
            )
            
            # Test (hold-out)
            df_test_holdout[f'{col}_encoded'] = df_test_holdout[col].astype(str).map(
                lambda x: encoder.transform([x])[0] if x in encoder.classes_ else -1
            )
            
            # Test (real - etiketli olmayan)
            if df_test_real is not None:
                df_test_real[f'{col}_encoded'] = df_test_real[col].astype(str).map(
                    lambda x: encoder.transform([x])[0] if x in encoder.classes_ else -1
                )
            
            encoders[col] = encoder
    
    return df_train, df_val, df_test_holdout, df_test_real, encoders

In [26]:
# ============================================================================
# ADIM 4: MODEL EĞİTİMİ
# ============================================================================

def train_model(df_train, df_val, threshold=0.5):
    """
    Logistic Regression eğitimi
    """
    
    # Feature seçimi
    numeric_features = [
        'week', 'month', 'year', 'day_of_year',
        'customer_age_days',
    ]
    
    # One-hot ve label encoded sütunları ekle
    onehot_cols = [col for col in df_train.columns if any(
        col.startswith(f'{cat}_') for cat in ['customer_category', 'customer_status', 'grade_name', 'unit_name']
    )]
    
    label_encoded_cols = [col for col in df_train.columns if col.endswith('_encoded')]
    
    features = numeric_features + onehot_cols + label_encoded_cols
    features = [f for f in features if f in df_train.columns]
    
    print("\n" + "="*80)
    print("MODEL EĞİTİMİ")
    print("="*80)
    print(f"Feature sayısı: {len(features)}")
    print(f"  - Numeric: {len(numeric_features)}")
    print(f"  - One-Hot: {len(onehot_cols)}")
    print(f"  - Label Encoded: {len(label_encoded_cols)}")
    
    # Veri hazırlama
    X_train = df_train[features].fillna(0)
    y_train = df_train['Target_purchase_next_1w']
    
    X_val = df_val[features].fillna(0)
    y_val = df_val['Target_purchase_next_1w']
    
    print(f"\nTarget dağılımı:")
    print(f"  Train - Pozitif: {y_train.sum():,} / {len(y_train):,} ({y_train.mean()*100:.2f}%)")
    print(f"  Val   - Pozitif: {y_val.sum():,} / {len(y_val):,} ({y_val.mean()*100:.2f}%)")
    
    # Model
    model = LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced',
        C=0.1  # Regularization
    )
    
    print(f"\nModel eğitiliyor...")
    model.fit(X_train, y_train)
    
    # Tahminler
    y_train_pred_proba = model.predict_proba(X_train)[:, 1]
    y_val_pred_proba = model.predict_proba(X_val)[:, 1]
    
    y_train_pred = (y_train_pred_proba > threshold).astype(int)
    y_val_pred = (y_val_pred_proba > threshold).astype(int)
    
    # Metrikler
    train_auc = roc_auc_score(y_train, y_train_pred_proba)
    val_auc = roc_auc_score(y_val, y_val_pred_proba)
    
    print("\n" + "="*80)
    print("VALİDATİON SONUÇLARI (Model Tuning İçin)")
    print("="*80)
    print(f"Train AUC: {train_auc:.4f}")
    print(f"Val AUC:   {val_auc:.4f}")
    print(f"Fark:      {abs(train_auc - val_auc):.4f}")
    
    if abs(train_auc - val_auc) > 0.05:
        print("  ⚠️  Overfitting var!")
    
    # Confusion Matrix
    cm = confusion_matrix(y_val, y_val_pred)
    print(f"\nValidation Confusion Matrix (threshold={threshold}):")
    print(cm)
    
    return model, features

In [27]:
# ============================================================================
# ADIM 5: TEST (HOLD-OUT) DEĞERLENDİRMESİ
# ============================================================================

def evaluate_on_test(df_test, model, features, threshold=0.5):
    """
    Hold-out test setinde final performansı ölç
    """
    
    print("\n" + "="*80)
    print("TEST (HOLD-OUT) DEĞERLENDİRMESİ")
    print("="*80)
    print("NOT: Bu skorlar validation'a overfitting'i kontrol eder")
    print("="*80)
    
    X_test = df_test[features].fillna(0)
    y_test = df_test['Target_purchase_next_1w']
    
    # Tahmin
    y_test_pred_proba = model.predict_proba(X_test)[:, 1]
    y_test_pred = (y_test_pred_proba > threshold).astype(int)
    
    # Metrikler
    test_auc = roc_auc_score(y_test, y_test_pred_proba)
    
    print(f"\nTest AUC: {test_auc:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_test_pred)
    print(f"\nTest Confusion Matrix:")
    print(cm)
    print(f"  - True Negatives:  {cm[0,0]:,}")
    print(f"  - False Positives: {cm[0,1]:,}")
    print(f"  - False Negatives: {cm[1,0]:,}")
    print(f"  - True Positives:  {cm[1,1]:,}")
    
    return test_auc

In [28]:
# ============================================================================
# ADIM 6: GERÇEK TEST (CANLI) TAHMİNİ
# ============================================================================

def predict_real_test(df_test_real, model, features, threshold=0.5):
    """
    Gerçek test seti (etiketli olmayan) için tahmin
    """
    
    print("\n" + "="*80)
    print("GERÇEK TEST (CANLI) TAHMİNİ")
    print("="*80)
    
    X_test_real = df_test_real[features].fillna(0)
    
    # Tahmin
    predictions_proba = model.predict_proba(X_test_real)[:, 1]
    predictions_binary = (predictions_proba > threshold).astype(int)
    
    df_result = df_test_real.copy()
    df_result['prediction_proba'] = predictions_proba
    df_result['prediction_binary'] = predictions_binary
    
    print(f"Tahmin sayısı: {len(predictions_proba):,}")
    print(f"Pozitif tahmin: {predictions_binary.sum():,} ({predictions_binary.mean()*100:.2f}%)")
    print(f"Ortalama olasılık: {predictions_proba.mean():.4f}")
    
    return df_result

In [29]:
# ============================================================================
# ANA PİPELİNE
# ============================================================================

def main_pipeline(df_train_raw, df_test_real_raw, val_weeks=4, test_weeks=4, threshold=0.5):
    """
    Tam pipeline:
    1. df_train_raw → Train + Val + Test (hold-out)
    2. df_test_real_raw → Canlı tahmin
    """
    
    print("="*80)
    print("ZAMAN SERİSİ TAHMİN PİPELİNE")
    print("="*80)
    
    # 1. Feature Engineering
    print("\n[1/7] Feature Engineering...")
    df_train = prepare_basic_features(df_train_raw)
    df_test_real = prepare_basic_features(df_test_real_raw)
    
    # 2. Train-Val-Test Split (3 parça!)
    print("\n[2/7] Train-Validation-Test Split...")
    df_train_split, df_val_split, df_test_holdout = split_train_val_test(
        df_train, val_weeks=val_weeks, test_weeks=test_weeks
    )
    
    # 3. Encoding
    print("\n[3/7] Kategorik encoding...")
    df_train_split, df_val_split, df_test_holdout, df_test_real, encoders = encode_categorical_features(
        df_train_split, df_val_split, df_test_holdout, df_test_real
    )
    
    # 4. Model Eğitimi (Train + Validation)
    print("\n[4/7] Model eğitimi...")
    model, features = train_model(df_train_split, df_val_split, threshold=threshold)
    
    # 5. Test (Hold-out) Değerlendirmesi
    print("\n[5/7] Test (hold-out) değerlendirmesi...")
    test_auc = evaluate_on_test(df_test_holdout, model, features, threshold=threshold)
    
    # 6. Gerçek Test Tahmini
    print("\n[6/7] Gerçek test (canlı) tahmini...")
    df_predictions = predict_real_test(df_test_real, model, features, threshold=threshold)
    
    # 7. Submission
    print("\n[7/7] Submission hazırlanıyor...")
    submission = df_predictions[['ID', 'prediction_proba']].copy()
    submission.columns = ['ID', 'Target_purchase_next_1w']
    
    print("\n" + "="*80)
    print("ÖZET")
    print("="*80)
    print(f"Validation AUC: {model.score(df_val_split[features].fillna(0), df_val_split['Target_purchase_next_1w']):.4f}")
    print(f"Test AUC: {test_auc:.4f}")
    print(f"\nSubmission hazır: {len(submission):,} satır")
    
    return model, features, submission, df_predictions

In [32]:
# ============================================================================
# KULLANIM
# ============================================================================


# Veriyi yükle

df_train_raw = pd.read_csv("../data/train.csv")
df_test_real_raw = pd.read_csv("../data/test.csv")

# Pipeline çalıştır
model, features, submission, predictions = main_pipeline(
    df_train_raw, 
    df_test_real_raw,
    val_weeks=4,
    test_weeks=4,
    threshold=0.5
)

# Submission kaydet
submission.to_csv('submission.csv', index=False)


ZAMAN SERİSİ TAHMİN PİPELİNE

[1/7] Feature Engineering...

[2/7] Train-Validation-Test Split...
TRAIN-VALIDATION-TEST SPLIT (3 PARÇA)

Toplam hafta sayısı: 46
İlk hafta: 2024-10-28T00:00:00.000000000
Son hafta: 2025-09-08T00:00:00.000000000

Split stratejisi:
  Validation başlangıcı: 2025-07-21T00:00:00.000000000
  Test başlangıcı: 2025-08-18T00:00:00.000000000

TRAIN SET:
  - Hafta sayısı: 38
  - Tarih aralığı: 2024-10-28 00:00:00 → 2025-07-14 00:00:00
  - Satır sayısı: 1,746,708
  - Oran: 82.6%

VALIDATION SET:
  - Hafta sayısı: 4
  - Tarih aralığı: 2025-07-21 00:00:00 → 2025-08-11 00:00:00
  - Satır sayısı: 183,864
  - Oran: 8.7%

TEST SET (Hold-out):
  - Hafta sayısı: 4
  - Tarih aralığı: 2025-08-18 00:00:00 → 2025-09-08 00:00:00
  - Satır sayısı: 183,864
  - Oran: 8.7%

NOT:
  - Train: Modeli eğitmek için
  - Validation: Hiperparametre optimize etmek için
  - Test: Final performansı ölçmek için (validation'a overfitting önleme)
  - Real Test (etiketli olmayan): Canlı tahmin için


In [33]:
submission.head()

,ID,Target_purchase_next_1w
0,438_278_20250922,0.265731
1,367_179_20250922,0.476501
2,637_130_20250922,0.033017
3,568_62_20250922,0.517127
4,667_168_20250922,0.241416


In [8]:
df_train_raw = pd.read_csv("../data/train.csv")

df_test_raw = pd.read_csv("../data/test.csv")

In [9]:
df_train_raw.tail()

,ID,customer_id,product_unit_variant_id,week_start,qty_this_week,num_orders_week,spend_this_week,purchased_this_week,product_id,grade_name,unit_name,product_grade_variant_id,selling_price,customer_category,customer_status,customer_created_at,Target_qty_next_1w,Target_purchase_next_1w,Target_qty_next_2w,Target_purchase_next_2w
2114431,587_29_20250908,587,29,2025-09-08,0.0,0.0,0.0,0,132,GRADE_01,UNIT_004,174,170.0,CUST_CAT_003,CUST_STAT_000,2025-04-12,0.0,0,0.0,0
2114432,719_42_20250908,719,42,2025-09-08,0.0,0.0,0.0,0,137,GRADE_01,UNIT_007,179,40.0,CUST_CAT_003,CUST_STAT_000,2025-08-27,0.0,0,0.0,0
2114433,433_271_20250908,433,271,2025-09-08,0.0,0.0,0.0,0,148,GRADE_01,UNIT_008,193,250.0,CUST_CAT_000,CUST_STAT_002,2024-04-08,0.0,0,0.0,0
2114434,699_476_20250908,699,476,2025-09-08,0.0,0.0,0.0,0,398,GRADE_01,UNIT_008,447,135.0,CUST_CAT_003,CUST_STAT_000,2025-08-11,0.0,0,0.0,0
2114435,589_213_20250908,589,213,2025-09-08,0.0,0.0,0.0,0,106,GRADE_01,UNIT_008,137,135.0,CUST_CAT_003,CUST_STAT_000,2025-04-13,0.0,0,0.0,0


In [10]:
df_test_raw.head()

,ID,customer_id,product_unit_variant_id,week_start,product_id,grade_name,unit_name,product_grade_variant_id,customer_category,customer_status,customer_created_at
0,438_278_20250922,438,278,2025-09-22,157,GRADE_01,UNIT_004,202,CUST_CAT_007,CUST_STAT_000,2024-06-28
1,367_179_20250922,367,179,2025-09-22,135,GRADE_01,UNIT_007,177,CUST_CAT_002,CUST_STAT_000,2023-09-29
2,637_130_20250922,637,130,2025-09-22,83,GRADE_01,UNIT_004,104,CUST_CAT_001,CUST_STAT_000,2025-05-29
3,568_62_20250922,568,62,2025-09-22,76,GRADE_01,UNIT_004,96,CUST_CAT_003,CUST_STAT_000,2025-03-12
4,667_168_20250922,667,168,2025-09-22,101,GRADE_01,UNIT_004,130,CUST_CAT_003,CUST_STAT_000,2025-06-25


In [ ]:
"""
Tüm süreci çalıştır
"""

print("="*70)
print("BASİT ZAMAN SERİSİ TAHMİN PİPELİNE")
print("="*70)

# 1. Feature Engineering
print("\n[1/6] Feature Engineering...")
df_train = prepare_basic_features(df_train_raw)
df_test = prepare_basic_features(df_test_raw)

# 2. Train-Validation Split
print("\n[2/6] Train-Validation Split...")
df_train_split, df_val_split = split_for_time_series(df_train, val_weeks=4)


# 3. Kategorik Encode
print("\n[3/6] Kategorik değişkenler encode ediliyor...")
df_train_split, df_val_split, df_test, encoders = encode_categorical_features(
    df_train_split, df_val_split, df_test
)

# 4. Model Eğitimi
print("\n[4/6] Model eğitimi...")
model, features = train_simple_model(df_train_split, df_val_split,0.5)


# 5. Test Tahmini
print("\n[5/6] Test tahmini...")
df_test_results = predict_test(df_test, model, features,0.5)

BASİT ZAMAN SERİSİ TAHMİN PİPELİNE

[1/6] Feature Engineering...

[2/6] Train-Validation Split...
TRAIN-VALIDATION SPLIT
Toplam hafta sayısı (week_start): 46
İlk hafta (week_start): 2024-10-28T00:00:00.000000000
Son hafta (week_start): 2025-09-08T00:00:00.000000000

Validation başlangıcı: 2025-08-18T00:00:00.000000000

Train:
  - Hafta sayısı: 42
  - İlk hafta: 2024-10-28 00:00:00
  - Son hafta: 2025-08-11 00:00:00
  - Satır sayısı: 1,930,572

Validation:
  - Hafta sayısı: 4
  - İlk hafta: 2025-08-18 00:00:00
  - Son hafta: 2025-09-08 00:00:00
  - Satır sayısı: 183,864

[3/6] Kategorik değişkenler encode ediliyor...

--- ENCODING STRATEJİSİ ---
One-Hot Encoding: ['customer_category', 'customer_status', 'grade_name', 'unit_name']
Label Encoding: ['customer_id', 'product_id']

--- ONE-HOT ENCODING ---

customer_category: 8 benzersiz değer
  Değerler: ['CUST_CAT_003', 'CUST_CAT_000', 'CUST_CAT_006', 'CUST_CAT_005', 'CUST_CAT_002']...

customer_status: 4 benzersiz değer
  Değerler: ['CUST_